In [ ]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [ ]:
!pip install -U ddgs # 덕덕고

In [ ]:
!pip install tavily-python wikipedia duckduckgo-search numexpr

In [ ]:
!pip install wikipedia

In [5]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm = ChatOpenAI(model="gpt-4o-mini")

In [14]:
from langchain_community.tools.tavily_search import TavilySearchResults
# Tavily는 api-key 다운 필요# Tavily는 api-key 다운 필요
  # 여기서 가져올것 https://app.tavily.com/
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')
search_tool = TavilySearchResults(max_results = 3)

from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

ddg_wrapper = DuckDuckGoSearchAPIWrapper(max_results=3)
ddg_tool = DuckDuckGoSearchResults(api_wrapper = ddg_wrapper)

## 🛠️ ToolWithFallback 클래스 설명

이 클래스는 메인 도구(Primary Tool)가 실패할 경우를 대비해 재시도(Retry) 및 보조 도구(Fallback) 기능을 제공하는 래퍼(Wrapper) 클래스입니다. 시스템의 안정성을 높이기 위해 사용됩니다.
1. 주요 구성 요소 (__init__)
  - primary: 우선적으로 사용할 메인 도구입니다.
  - fallback: 메인 도구가 실패했을 때 대신 사용할 보조 도구입니다.
  - max_retries: 메인 도구의 최대 재시도 횟수입니다. (기본값 2회)
  - stats: 도구 호출의 성공 및 실패 횟수를 기록하는 상태 저장소입니다.
2. 동작 로직 (invoke)
  - 메인 도구 시도: max_retries만큼 반복하며 primary 도구를 호출합니다.
  - 성공 시: 결과를 즉시 반환하고 통계를 업데이트합니다.
  - 실패 시: 에러 메시지를 출력하고 다음 횟수로 넘어갑니다.
  - 보조 도구(Fallback) 실행: 모든 재시도가 실패하고 fallback 도구가 설정되어 있다면 실행됩니다.
  - 입력값(args)이 딕셔너리 형태일 경우 'query' 키를 추출하여 전달하는 유연성을 갖추고 있습니다.
  - 성공 시 어떤 도구로 결과가 나왔는지 표시하며 반환합니다.

💡 코드의 특징 (Q&A)
Fallback은 언제 실행되나요?
메인 도구가 max_retries만큼 반복했음에도 불구하고 계속 에러가 발생할 때 마지막 수단으로 실행됩니다.
왜 쓰나요?
특정 API(예: 구글 검색)가 할당량 초과나 네트워크 오류로 응답하지 않을 때, 서비스가 중단되지 않고 다른 대안(예: 덕덕고 검색)을 통해 결과를 얻기 위함입니다.
주의사항: 코드 내 __init__ 함수명에 언더바(_)가 빠져있으므로 실사용 시에는 def __init__(self, ...):로 수정이 필요합니다.

In [19]:
class ToolWithFallback:
    def __init__(self, primary, fallback=None, max_retries=2):
        self.primary = primary
        self.fallback = fallback
        self.max_retries = max_retries
        self.stats = {'primary_ok' : 0, 'fallback_ok': 0, 'errors':0}

    def invoke(self, args):
      for attempt in range(self.max_retries):
        try:
          result = self.primary.invoke(args)
          if result and str(result).strip():
            self.stats['primary_ok'] += 1
            return result

        except:
          print(f"{self.primary.name} error (attempt {attempt+1})")

      if self.fallback: # fallback을 어떻게 씀? ToolWithFallback(search_tool, ddg_tool) 즉, search_tool이 안되면 ddg_tool을 써라 라는것
        try:
          fallback_args = args.get('query', args) if isinstance(args, dict) else args
          result = self.fallback.invoke(fallback_args)
          self.stats['fallback_ok'] += 1
          return f"fallback: {self.fallback.name} | {result}"
        except:
          print(f"fallback fail too")


      self.stats['errors'] += 1

      return 'no result'

In [20]:
search_result = ToolWithFallback(primary = search_tool, fallback = ddg_tool)

In [21]:
search_result

In [ ]:
@tool
def search(query):
  """query가 들어가서 검색을 해오는 툴"""
  return f"{query} 입니다."

## MCP 예시 파일 설명

In [ ]:
# 툴을 서버 단위로 확장한 것이 MCP임
# 유저들이 물어보고, 서버에서 답변하는 플로우가 어려움
  # 예를 들어 "ETF 검색 툴을 만들었다" 라고 할때 다른사람들이 쓰게 하고싶다면
  # 우리가 만든 코드를 갖고있지 않으면 쓰기 쉽지 않음 -> 이걸 서버에 올려야함

# 도구가 기하급수적으로 늘어나고, 사용하는 서비스/사용자가 늘어나게 된다면
  # 사용자 * 도구 => N*M개의 커넥터가 필요 == 비효율적

# MCP는 프로토콜, 즉 규칙을 알려줄테니 규칙대로만 해
  # 툴(a)도 MCP에 연결
  # 유저(b)도 MCP에 연결
  # 즉 a+b개 만큼의 커넥터? 만 있으면 됨

In [ ]:
# 01_basic_server.py 설명
  # from mcp.server.fastmcp import FastMCP // FastMCP는 MCP 서버를 만드는 라이브러리
    # api(또는 라이브러리)
    # mcp = FastMCP('서버명') // 객체 생성 (서버)
      # 실제 사용시에는 DB(외부/내부)도 이용해야함 (코드에서는 단순히 딕셔너리 등)

  # @mcp.tool()  ---> 툴을 서버에 띄워놓고 이용하면 LLM(chatbot)이 보다가 툴에서 값을 받아와 전달
    # 해당 파일에 툴들이 정의되어있음
    # 함수를 데코레이터를 활용해 mcp tool로 만들어줌

  # 내부적으로는 json 형태로 전달된다고 함...
    # 툴들을 묶어서 전달해줌, 입력/출력을 정해진 규칙대로 전해줌
    # 예:
    # { 'name' : search etf,
    #   'args' : {'category':...
    #   }  }
  # mcp.run() 등으로 서버를 띄워놓게 함

# ====================================================================================

# 01_basic_client.py 설명
  # 서버에 연결해서 tool들을 하나하나씩 실행을 하는 파일임

  # import asyncio ---> 툴 호출할때 비동기식으로 작성
    # async def main():...
      # 어떤 툴(함수)은 빨리 끝날수 있고 다른건 늦게 끝날 수 있음
        # cpu를 계속 잡아먹고있게됨..
      # async를 사용하면 기다리는동안 다른 작업을 수행한다던지 할 수 있음
    # StdioServerParameter 클래스를 통해서, mcp 서버를 실행함
      # ...args = ['servers/01_basic_server.py']...

  # async with stdio_client(server_params) ...:
    # 핸드 셰이크 await session.initiolize() // 즉 초기화용 정보 교환 등
  # session.list_tools() // 사용 가능한 도구, 툴 이름, 설명 등

  # session.call_tool( "툴 이름", argument={"category": ... "min_return" : ...})
    # 서버측에서 정한 툴의 규칙에 따라 작성..

# ====================================================================================

# 02_langchain_client.py 설명
  # 01 파일과 유사하지만 langchain으로 감싸서 활용함

  # 거의 똑같지만, llm.bind_tools(툴)

  # inputSchema.get("properties", ..) // 파라미터 정보 호출

## 외부 api 활용
지금은 langchain 내부에 있는 툴이나, 간단한 툴들을 만들어 썼는데, 외부 api를 이용해서 LLM 서비스 붙이는 것을 할 예정

In [27]:
import os, json, time, hmac, hashlib, requests
#from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
#load_dotenv()

BASE_URL = "https://jsonplaceholder.typicode.com" # 연습용 api 조회, 값 넣고 등등 flow를 테스트 할 수 있는 사이트

### api
- 날씨 api, 슬랙(slack), 등등.. 대부분의 서비스들이 api를 제공함
- llm이 api들을 쓰게 해주는 코드를 작성할 예정
- RestAPI로 가져오고 값을 넣고 등, 같이 활용해야해서 어려울 수 있음

- HTTP : 클라이언트 -> 요청을 보냄 (request)
  - 서버 -> 응답 (response)

In [22]:
# request : 랭체인 설명해줘
  # Method : 뭘 할건지? POST/GET...요청/PUT 삽입 /DELETE 삭제
  # URL : 어디서 할건지? 예) www.naver.com
  # 헤더 : 니가 누군데? 예) {인증정보, 브라우저(chrome/mozila...)}
  # Body : 뭘 보낼건지? 예) 로그인을 한다면 id 등 {text : abced}


In [ ]:
# 요청 200, 201, 400, 401, 403, 500 ..
  # 2로 시작하면 OK라는 뜻
    # 200 조회 성공
    # 201 created 성공
  # 4는 실패
    # 400: 요청 형식 틀림
    # 401: unauthorized
    # 403: forbidden
  # 5 -> 500 internal server error

In [31]:
requests.get(f"{BASE_URL}/posts/1")

<Response [200]>

In [29]:
BASE_URL = "https://jsonplaceholder.typicode.com"
# https://jsonplaceholder.typicode.com/posts/1 에 접속하면
# 아래 확인가능
# {
#   "userId": 1,
#   "id": 1,
#   "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
#   "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
# }

In [35]:
r = requests.post(f"{BASE_URL}/posts", json = {'title': "새 글", 'body' : '본문', 'userid': 1})


In [36]:
r.status_code

201

In [37]:
r.json()

{'title': '새 글', 'body': '본문', 'userid': 1, 'id': 101}

In [39]:
r = requests.delete(f"{BASE_URL}/posts/1", timeout = 5) # timeout 등, 너무 오래 걸리면 중단
r

<Response [200]>

In [42]:
session = requests.Session() # 세션을 열어줌
session.headers.update({
    'User-Agent' : 'abcde/1.0', # 아무거나 넣어도 되지만, 실제 서비스에서는 Mozila, Chrome 등 신뢰성 높은 것만 받기도함
    'Accept' : 'application/json'

})

In [44]:
r = session.get(f"{BASE_URL}/posts", params = {"userid1":1})
r

<Response [200]>

In [45]:
r.json() # posts 전체를 get해옴

[{'userId': 1,
  'id': 1,
  'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
  'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'},
 {'userId': 1,
  'id': 2,
  'title': 'qui est esse',
  'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'},
 {'userId': 1,
  'id': 3,
  'title': 'ea molestias quasi exercitationem repellat qui ipsa sit aut',
  'body': 'et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut'},
 {'userId': 1,
  'id': 4,
  'title': 'eum et est occaecati',
  'body': 'ullam et saepe reiciendis voluptatem adipisci\nsit amet autem assumenda provid

In [48]:
r2 = session.get(f"{BASE_URL}/users/1")
r2.json()['name'], r2.json()['email']

('Leanne Graham', 'Sincere@april.biz')

In [47]:
r2

<Response [200]>

In [ ]:
# 세션 사용해서 rank_user_by_post 라는 함수를 만들고, user_ids 리스트를 입력으로 받은 후
  # return 유저들인데, 각각 게시글을 몇개 썼는지 등등

""" 예시
[{'userId': 1,
  'id': 1,
  'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
  'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'},
 {'userId': 1,
  'id': 2,
  'title': 'qui est esse',
  'body': 'est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil molestiae ut reiciendis\nqui aperiam non debitis possimus qui neque nisi nulla'},
 {'userId': 1,
  'id': 3,
  'title': 'ea molestias quasi exercitationem repellat qui ipsa sit aut',
  'body': 'et iusto sed quo iure\nvoluptatem occaecati omnis eligendi aut ad\nvoluptatem doloribus vel accusantium quis pariatur\nmolestiae porro eius odio et labore et velit aut'},
 {'userId': 1,
  'id': 4,
  'title': 'eum et est occaecati',
  'body': 'ullam et saepe reiciendis voluptatem adipisci\nsit amet autem assumenda provident rerum culpa\nquis hic commodi nesciunt rem tenetur doloremque ipsam iure\nquis sunt voluptatem rerum illo velit'},""
  """
def rank_users_by_posts(user_ids):
  session.get(f"{BASE_URL}/users/1")

  여기서 어떻게 가져오지?


  return



#### 내 코드

In [57]:
BASE_URL = "https://jsonplaceholder.typicode.com"
session = requests.Session()
# 현재 코드는 for문을 돌면서 유저가 10명이면 10번, 100명이면 100번 서버에 요청을 보냄
def rank_users_by_posts(user_ids):
    user_post_counts = []

    for user_id in user_ids:
        response = session.get(f"{BASE_URL}/posts", params={"userId": user_id})
        posts = response.json()
        post_count = len(posts)

        user_post_counts.append({
            "userId": user_id,
            "post_count": post_count,
        })

    ranked_users = sorted(user_post_counts, key=lambda x: x['post_count'], reverse=True)

    return ranked_users

In [58]:
rank_users_by_posts([1,2,3])

[{'userId': 1, 'post_count': 10},
 {'userId': 2, 'post_count': 10},
 {'userId': 3, 'post_count': 10}]

#### 강사님 작성

In [60]:
def rank_users_by_posts(user_ids):
  s = requests.Session()
  session.headers.update({
      'User-Agent' : 'abcde/1.0',
      'Accept' : 'application/json'
  })

  counts = []
  for uid in user_ids:
    r = session.get(f"{BASE_URL}/posts", params = {"userid":uid})
    # r.json으로 하면 uid 번 유저가 남긴 총 개수가 됨
    counts.append({"userid" : uid, "post_count" : len(r.json())})

  counts.sort(key=lambda x : x['post_count'], reverse = True)
  for i, item in enumerate(counts, 1):
    item['rank'] = i

  return counts

In [61]:
rank_users_by_posts([1,2,3])

[{'userid': 1, 'post_count': 100, 'rank': 1},
 {'userid': 2, 'post_count': 100, 'rank': 2},
 {'userid': 3, 'post_count': 100, 'rank': 3}]

### LLM과 API 연동
- tool로 감싸줘야함

In [ ]:
# {BASE_URL}/users/i 예시

# {
#   "id": 1,
#   "name": "Leanne Graham",
#   "username": "Bret",
#   "email": "Sincere@april.biz",
#   "address": {
#     "street": "Kulas Light",
#     "suite": "Apt. 556",
#     "city": "Gwenborough",
#     "zipcode": "92998-3874",
#     "geo": {
#       "lat": "-37.3159",
#       "lng": "81.1496"
#     }
#   },
#   "phone": "1-770-736-8031 x56442",
#   "website": "hildegard.org",
#   "company": {
#     "name": "Romaguera-Crona",
#     "catchPhrase": "Multi-layered client-server neural-net",
#     "bs": "harness real-time e-markets"
#   }

In [83]:
# 실제 사용시에는 API 문서들을 읽어보면서 작성해야함.
  # 구조 등등..
@tool
def fetch_user(user_id): # user_id를 넣으면 특정 유저의 정보를 가져오겠다..
  """지정된 ID의 사용자 정보를 조회합니다."""
  r = requests.get(f"{BASE_URL}/users/{user_id}")
  if not r.ok: # 즉 True / False
    return f"error: status {r.status_code}"

  data = r.json()
  return json.dumps({
      'name' : data['name'],
      'email' : data['email'],
      'phone' : data['phone'],
      'company' : data['company']['name']
  }, ensure_ascii=False)

@tool
def fetch_user_post(user_id):
  """지정된 ID의 사용자가 작성한 게시글을 조회합니다."""
  r = requests.get(f"{BASE_URL}/posts", params={'userId': user_id})
  if not r.ok:
    return f"error: status {r.status_code}"
  posts = r.json()
  return json.dumps([{'id' : p['id'], 'title': p['title']} for p in posts], ensure_ascii = False)

In [84]:
fetch_user(1)

TypeError: 'StructuredTool' object is not callable

In [85]:
fetch_user_post(1)

TypeError: 'StructuredTool' object is not callable

In [86]:
llm_api = llm.bind_tools([fetch_user, fetch_user_post])
response = llm_api.invoke("1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘")
print(f"호출된 도구 개수: {len(response.tool_calls)}")

호출된 도구 개수: 2


In [87]:
for tc in response.tool_calls:
  print(f"{tc['name']} : {tc['args']}")

# "1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘" 에 대한 정보

fetch_user : {'user_id': 1}
fetch_user_post : {'user_id': 1}


In [89]:
tool_map = {'fetch_user': fetch_user, 'fetch_user_post' : fetch_user_post}
messages = [HumanMessage(content = '1번 유저 정보와 그 사람이 쓴 게시글 갯수 알려줘')]
resp = llm_api.invoke(messages)
messages.append(resp)

for tc in resp.tool_calls:
  result = tool_map[tc['name']].invoke(tc['args'])
  messages.append(ToolMessage(content=result, tool_call_id = tc['id']))

final = llm_api.invoke(messages)

In [90]:
final

AIMessage(content='1번 유저의 정보는 다음과 같습니다:\n\n- 이름: Leanne Graham\n- 이메일: Sincere@april.biz\n- 전화번호: 1-770-736-8031 x56442\n- 회사: Romaguera-Crona\n\n이 유저가 쓴 게시글은 총 10개입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 383, 'total_tokens': 454, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_38babba494', 'id': 'chatcmpl-DXnsBY3s1MlFTG5WYIpnZ3DckmN7T', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019dba68-8f42-7333-a526-687e3d93a527-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 383, 'output_tokens': 71, 'total_tokens': 454, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio':

In [100]:
r = requests.get(f"{BASE_URL}/todos?userId=1")

In [102]:
len(r.json())

20

In [105]:
# https://jsonplaceholder.typicode.com/todos/?userId=1

def fetch_todos(user_id) :
  r = requests.get(f"{BASE_URL}/todos?userId={user_id}")
  total_count = len(r.json())
  top_5 = r.json()[:5]

  return { # 지정된 id 사용자의 todo 목록 상위 5개
        "total_count": total_count,
        "top_5": top_5
    }

def count_completed_todos(user_id):
    r = requests.get(f"{BASE_URL}/todos", params={'userId': user_id})

    all_todos = r.json()
    total = len(all_todos)
    completed_count = len([t for t in all_todos if t['completed'] is True])
    completion_rate = (completed_count / total * 100) if total > 0 else 0

    return {
        'user_id': user_id,
        'completed': completed_count,
        'total': total,
        'completion_rate': f"{completion_rate:.1f}%"
    }


In [107]:
fetch_todos(1)

{'total_count': 20,
 'top_5': [{'userId': 1,
   'id': 1,
   'title': 'delectus aut autem',
   'completed': False},
  {'userId': 1,
   'id': 2,
   'title': 'quis ut nam facilis et officia qui',
   'completed': False},
  {'userId': 1, 'id': 3, 'title': 'fugiat veniam minus', 'completed': False},
  {'userId': 1, 'id': 4, 'title': 'et porro tempora', 'completed': True},
  {'userId': 1,
   'id': 5,
   'title': 'laboriosam mollitia et enim quasi adipisci quia provident illum',
   'completed': False}]}

In [112]:
count_completed_todos(1)

{'user_id': 1, 'completed': 11, 'total': 20, 'completion_rate': '55.0%'}

In [114]:
# 강사님 코드

@tool
def fetch_todos(user_id) :
  """지정된 사용자의 todo 목록 상위 5개를 조회합니다."""
  r = requests.get(f"{BASE_URL}/todos", params = {'userID' : user_id})
  if not r.ok:
    return f"error : {r.status_code}"
  todos = r.json()[:5]

  return  json.dumps(todos, ensure_ascii=False)

@tool
def count_completed_todos(user_id):
  """지정된 사용자의 완료된 todo 개수와 전체 개수를 계산합니다."""
  r = requests.get(f"{BASE_URL}/todos", params = {'userID' : user_id})
  if not r.ok:
    return f"error: {r.status_code}"

    todos = r.json()
    completed = sum(1 for t in todos if t['completed'])

    total = len(todos)
    return json.dumps(
    {
        'user_id': user_id,
        'completed' : completed,
        'total' : total,
        'completion_rate' : round(completed/total * 100, 1) if total else 0,
    }, ensure_ascii= False)

In [122]:
todo_tools = [fetch_todos, count_completed_todos]
llm_todo = llm.bind_tools(todo_tools)
tool_map = {t.name:t for t in todo_tools}

In [123]:
messages = [HumanMessage(content = '1번 유저의 todo 완료율을 알려줘')]
resp = llm_api.invoke(messages)
messages.append(resp)

print(f"tools: {[tc['name'] for tc in resp.tool_calls]}")

for tc in resp.tool_calls:
  result = tool_map[tc['name']].invoke(tc['args'])
  messages.append(ToolMessage(content=result, tool_call_id = tc['id']))

final = llm_api.invoke(messages)

tools: ['fetch_user']


KeyError: 'fetch_user'